Characterize functions for sparse undirected graphs up to 100.000 nodes

In [1]:
import numpy as np
from igraph import Graph
from leidenalg import find_partition, ModularityVertexPartition


def characterize_network_scalable(
    G,
    outfile=None,
    size_density=1,
    degree_stats=1,
    clustering=1,
    shortest_path=1,
    assortativity=1,
    centralities=1,
    communities=1,
    
):
    results = {}

    # ==========================================================
    # 1. Size and density
    # ==========================================================
    if size_density:
        results["N"] = G.vcount()
        results["L"] = G.ecount()
        results["density"] = G.density()

    # ==========================================================
    # 2. Degree distribution
    # ==========================================================
    if degree_stats:
        degrees = np.array(G.degree())
        results["k_avg"] = np.mean(degrees)
        results["k_var"] = np.var(degrees)
        results["degrees"] = degrees

    # ==========================================================
    # 3. Clustering coefficient
    # ==========================================================
    
    if clustering:
        print("3-clustering")
        clust = G.transitivity_local_undirected(vertices=None)
        clust = [c for c in clust if not np.isnan(c)]
        results["clustering"] = float(np.mean(clust)) if clust else np.nan

    # ==========================================================
    # 4. Average shortest path
    # ==========================================================
    if shortest_path:
        N = G.vcount()

        if G.is_connected():
           
            results["avg_shortest_path"] = G.average_path_length()
        else:
            gcc = G.connected_components().giant()
            Ngcc = gcc.vcount()
            results["gcc_size"] = Ngcc
            results["avg_shortest_path"] = gcc.average_path_length()
    # ==========================================================
    # 5. Assortativity
    # ==========================================================
    if assortativity:
        print("5-assortativity")

        results["assortativity"] = G.assortativity_degree()

    
        
    # ==========================================================
    # 6. Centralities (undirected)
    # ==========================================================
    if centralities:
        print("6-centralities",flush=True)

        N = G.vcount()
        E = G.ecount()

        # 1. Degree centrality 
        results["degree_centrality"] = dict(enumerate(G.degree()))

        # 2. Betweenness centrality
        
        
        bw = G.betweenness(directed=False)
        results["betweenness_centrality"] = dict(enumerate(bw))
        

        # 3. Eigenvector centrality 
        
        
        eig = G.eigenvector_centrality()
        results["eigenvector_centrality"] = dict(enumerate(eig))
        

        #4. Page rank

        pr = G.pagerank(directed=False)
        results["pagerank"] = dict(enumerate(pr))
        

            

    # ==========================================================
    # 7. Communities (Leiden)
    # ==========================================================
    if communities:
            print("7-communities")
            # for huge graphs we work on the giant component
            gcc = G.connected_components().giant()
            partition = find_partition(gcc, ModularityVertexPartition)
            results["n_communities"] = len(partition)
            results["community_sizes"] = sorted([len(c) for c in partition], reverse=True)
            results["modularity"] = partition.modularity
            results["communities_on_gcc"] = True
            results["gcc_size_for_communities"] = gcc.vcount()

    # ==========================================================
    # Save to file
    # ==========================================================

    if outfile is not None:
            with open(outfile, "w") as f:
                for key, value in results.items():
                    if key == "degrees":
                        continue
                    if isinstance(value, dict):
                        f.write(f"{key}\n")
                        for k, v in value.items():
                            f.write(f"    {k}: {v}\n")
                    else:
                        f.write(f"{key}: {value}\n")

            if degree_stats:
                degrees_outfile = outfile.replace(".txt", "_degrees.txt")
                np.savetxt(degrees_outfile, degrees, fmt="%d")

    

    return results


In [2]:
import pandas as pd
import igraph as ig

# 1. Read the file
edges = pd.read_csv("edge_list_PG.csv",
                    header=None, usecols=[0,1],
                    names=["src", "dst"])

# 2. Remove self loops
edges = edges[edges["src"] != edges["dst"]]

# 3. If ther file is directed, to make it undirected, order couples
edges_und = edges.apply(lambda row: tuple(sorted((row["src"], row["dst"]))), axis=1)

# 4. Remove dupicates
edges_und = list(set(edges_und))

# 5. Creates clean igraph
G = ig.Graph.TupleList(edges_und, directed=False)



res = characterize_network_scalable(G, "PG_results.txt")

3-clustering
5-assortativity
6-centralities


C:\Users\VALE\AppData\Local\Temp\ipykernel_16508\2755561434.py:93: RuntimeWarning: Some eigenvector centralities are nearly zero, indicating that the graph may not be (strongly) connected. Eigenvector centrality is not meaningful for disconnected graphs. Location: src/centrality/eigenvector.c:103
  eig = G.eigenvector_centrality()


7-communities
